# Практика · Аугментація даних

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає вісім мереж — на них іде майже весь його час. Заміряно на чотирьох ядрах
> без відеокарти, в один потік: від 85 до 165 секунд, залежно від того, чим ще зайнята
> машина. Якщо прибрати `torch.set_num_threads(1)`, час зростає в рази.

У лекції ми стверджували, що аугментація — це не «більше картинок», а заява про
інваріантність, і що заява буває хибною. Тут ти перевіриш кожне з тих тверджень числом.

Що зробимо:

1. **Намалюємо датасет** — шість класів фігур 28×28, формулами, без завантажень.
2. **Напишемо дзеркало руками** на numpy й переконаємось, що воно **точно** збігається
   з `torchvision.transforms.v2`.
3. **Поміряємо геометрією**, яке перетворення зберігає мітку, а яке ні — без жодного навчання.
4. **Замір 1:** виграш аугментації на маленькій вибірці (120 зображень) і розрив між
   навчальною й перевірочною точністю.
5. **Замір 3:** шкода від надто сильного повороту — той самий конвеєр, змінено одне число.
6. **Замір 2:** той самий дослід на 360 і 1200 прикладах — і що стається з виграшем.
7. **Замір 4:** що буде, якщо аугментувати **перевірочну** вибірку.
8. **TTA** — усереднення прогнозів по кількох варіантах і його ціна.

**Мережа не потрібна:** датасет ми малюємо самі, формулами.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
from torchvision.transforms import v2

# зерна фіксуємо на самому початку: без них числа нижче не збіжаться з лекцією
torch.manual_seed(0)
rng = np.random.default_rng(42)

# один потік — не заради швидкості, а заради відтворюваності: під кількома потоками
# float-додавання йде в іншому порядку, суми виходять інші, і числа зошита перестають
# збігатися з лекцією. На мережах у тисячі ваг багатопотоковість усе одно нічого не дає.
torch.set_num_threads(1)

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Датасет: шість фігур, намальованих формулами

Той самий генератор, що й у попередніх темах блоку, але з більшим розкидом: центр гуляє
на ±4 пікселі, радіус від 5 до 8, шум 0.20. Без цього розкиду задача виявляється надто
легкою, і аугментації нема чого рятувати.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=4, noise=0.20):
    """Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1."""
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо на кілька пікселів, щоб мережа не завчила одне положення
    center_y = (size - 1) / 2 + rng.integers(-jitter, jitter + 1)
    center_x = (size - 1) / 2 + rng.integers(-jitter, jitter + 1)
    radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 2.5) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 1.5) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 1.5) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_shape_dataset(count, rng):
    """Повертає тензори (count, 1, 28, 28) і (count,) з рівною кількістю класів."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


started = time.time()
pool_x, pool_y = make_shape_dataset(1200, rng)      # з нього беремо перші 120 / 360 / 1200
test_x, test_y = make_shape_dataset(600, rng)       # перевірочна — одна на весь зошит
print("згенеровано за %.2f с" % (time.time() - started))
print("запас навчальних:", tuple(pool_x.shape))
print("перевірочна     :", tuple(test_x.shape))

Подивимось на фігури очима — інакше далі буде незрозуміло, що саме мережа розрізняє.
Зверни увагу на **квадрат і ромб**: далі виявиться, що це той самий предмет, повернутий
на 45°, і саме на цьому тримається половина теми.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 6, figsize=(11, 2))
for kind in range(6):
    # перші шість зображень датасету — саме по одному на кожен клас
    axes[kind].imshow(pool_x[kind, 0], cmap="gray")
    axes[kind].set_title(SHAPE_NAMES[kind], fontsize=10)
    axes[kind].axis("off")
plt.tight_layout()
plt.show()
print("шість класів, по", len(pool_y) // 6, "прикладів кожного в запасі")

## 2 · Наше дзеркало проти бібліотечного

Перше правило курсу: перш ніж користуватись бібліотекою, зроби те саме руками й звір.
Дзеркало по горизонталі — це просто перестановка стовпців у зворотному порядку, тобто
зріз `[..., ::-1]`. Порівняємо з `v2.functional.horizontal_flip` і вимагатимемо **точного**
збігу, а не приблизного: перестановка не втрачає жодного біта.

In [ ]:
def flip_horizontal_by_hand(images):
    """Дзеркало по горизонталі: останній вимір (стовпці) у зворотному порядку."""
    as_numpy = images.numpy()
    # ::-1 дає «вид» із відʼємним кроком, а torch такого не приймає — робимо копію
    flipped = np.ascontiguousarray(as_numpy[..., ::-1])
    return torch.from_numpy(flipped)


our_flip = flip_horizontal_by_hand(pool_x[:64])
library_flip = v2.functional.horizontal_flip(pool_x[:64])

assert torch.equal(our_flip, library_flip), "дзеркало розійшлося з бібліотечним!"
print("✅ збігається точно, до останнього біта")
print("максимальна різниця:", (our_flip - library_flip).abs().max().item())

Друге порівняння — складніше й тому цікавіше. Зсув на ціле число пікселів теж можна
написати руками: беремо `np.roll` і зануляємо те, що «заїхало» з протилежного краю.
`torchvision` робить це через афінне перетворення, і при цілому зсуві та найближчому
сусіді результат мусить збігтися.

In [ ]:
def shift_by_hand(images, dy, dx):
    """Зсув на ціле число пікселів; те, що виїхало за край, заміняємо нулями."""
    moved = torch.roll(images, shifts=(dy, dx), dims=(2, 3))
    # roll переносить пікселі по колу, а нам потрібен порожній край — зануляємо його
    if dy > 0:
        moved[:, :, :dy, :] = 0
    elif dy < 0:
        moved[:, :, dy:, :] = 0
    if dx > 0:
        moved[:, :, :, :dx] = 0
    elif dx < 0:
        moved[:, :, :, dx:] = 0
    return moved


our_shift = shift_by_hand(pool_x[:64], dy=3, dx=-2)
library_shift = v2.functional.affine(
    pool_x[:64], angle=0.0, translate=[-2, 3], scale=1.0, shear=[0.0, 0.0],
    interpolation=v2.InterpolationMode.NEAREST)

print("однакових пікселів: %.4f" % torch.isclose(our_shift, library_shift).float().mean().item())
assert torch.allclose(our_shift, library_shift), "зсув розійшовся з бібліотечним!"
print("✅ збігається: np.allclose пройшов")

## 3 · Яке перетворення зберігає мітку — геометрією, без навчання

Тепер найважливіший інструмент теми. Щоб перевірити твердження
«`y(T(x)) = y(x)`», навчати мережу не потрібно: досить порівняти перетворену фігуру
з еталонними шаблонами всіх шести класів.

Три деталі, без яких вимірювання було б неправильним:

- **центруємо за центром мас** — інакше «зсув зламав мітку» означало б лише, що фігура
  поїхала вбік;
- **шаблон свого класу беремо з найближчою площею** — інакше «масштаб зламав мітку»
  означало б лише, що фігура стала більшою;
- **малюємо на сітці 56×56** замість 28×28 — на грубому растрі поворот лишає зубці, і
  збіг падає через них, а не через форму.

Міра збігу — IoU: площа спільної частини, поділена на площу обʼєднання.

In [ ]:
GRID = 56                      # удвічі дрібніша сітка: міряємо геометрію, а не зубці
CENTER = (GRID - 1) / 2
TEMPLATE_RADII = [8, 10, 12, 14, 16, 18, 20]
BASE_RADIUS = 14               # радіус фігури, з якою працюємо в цьому розділі


def shape_mask(kind, radius, size=GRID):
    """Та сама формула, що й у датасеті, але без шуму й на заданій сітці."""
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - (size - 1) / 2, xx - (size - 1) / 2
    if kind == 0:
        mask = dy * dy + dx * dx <= radius * radius
    elif kind == 1:
        mask = (np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)
    elif kind == 2:
        mask = np.abs(dy) + np.abs(dx) <= radius
    elif kind == 3:
        distance = dy * dy + dx * dx
        mask = (distance <= radius * radius) & (distance >= (radius - 5.0) ** 2)
    elif kind == 4:
        mask = (((np.abs(dy) <= 3.0) & (np.abs(dx) <= radius))
                | ((np.abs(dx) <= 3.0) & (np.abs(dy) <= radius)))
    else:
        mask = ((dy >= -radius * 0.8) & (dy <= radius * 0.8)
                & (np.abs(dx) <= (dy + radius * 0.8) * 0.6))
    return mask


def warp(mask, degrees=0.0, shift_y=0.0, shift_x=0.0, scale=1.0, flip=False):
    """Поворот, зсув, масштаб і дзеркало зворотним відображенням, найближчий сусід."""
    angle = np.deg2rad(degrees)
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    yy, xx = np.mgrid[0:GRID, 0:GRID]
    dy = (yy - CENTER - shift_y) / scale
    dx = (xx - CENTER - shift_x) / scale
    source_y = dy * cos_a - dx * sin_a
    source_x = dy * sin_a + dx * cos_a
    if flip:
        source_x = -source_x
    source_y = np.rint(source_y + CENTER).astype(int)
    source_x = np.rint(source_x + CENTER).astype(int)
    inside = ((source_y >= 0) & (source_y < GRID) & (source_x >= 0) & (source_x < GRID))
    out = np.zeros((GRID, GRID), dtype=bool)
    out[inside] = mask[source_y[inside], source_x[inside]]
    return out


def center_by_mass(mask):
    """Зсуває фігуру так, щоб її центр мас опинився в центрі кадру."""
    if not mask.any():
        return mask
    ys, xs = np.nonzero(mask)
    shift_y = int(round(CENTER - ys.mean()))
    shift_x = int(round(CENTER - xs.mean()))
    out = np.zeros_like(mask)
    new_y, new_x = ys + shift_y, xs + shift_x
    keep = (new_y >= 0) & (new_y < GRID) & (new_x >= 0) & (new_x < GRID)
    out[new_y[keep], new_x[keep]] = True
    return out


def iou(a, b):
    """Площа перетину, поділена на площу обʼєднання."""
    union = np.logical_or(a, b).sum()
    return 0.0 if union == 0 else float(np.logical_and(a, b).sum()) / union


TEMPLATES = {kind: [(center_by_mass(shape_mask(kind, r)), int(shape_mask(kind, r).sum()))
                    for r in TEMPLATE_RADII]
             for kind in range(6)}


def score_for(centered, area, kind):
    """Збіг із шаблоном свого класу, найближчим за площею."""
    template, _ = min(TEMPLATES[kind], key=lambda pair: abs(pair[1] - area))
    return iou(centered, template)


def nearest_class(mask):
    """Клас, на який перетворена фігура тепер схожа найбільше, і сам збіг."""
    centered = center_by_mass(mask)
    area = int(centered.sum())
    scores = [score_for(centered, area, kind) for kind in range(6)]
    best = int(np.argmax(scores))
    return best, scores[best]


# перевірка самого інструмента: неперетворена фігура мусить впізнаватись ідеально
for kind in range(6):
    got, score = nearest_class(shape_mask(kind, BASE_RADIUS))
    assert got == kind and score > 0.99, "класифікатор шаблонів зламаний на %s" % SHAPE_NAMES[kind]
print("✅ шаблонний класифікатор упізнає всі шість класів зі збігом 1.000")

Інструмент працює. Тепер поставимо йому три питання, які й вирішують, що можна класти
в конвеєр аугментації, а що ні.

In [ ]:
print("%-11s %-22s %-22s %s" % ("клас", "дзеркало", "зсув на 8", "поворот на 45°"))
print("-" * 78)
for kind in range(6):
    base = shape_mask(kind, BASE_RADIUS)
    results = []
    for transformed in (warp(base, flip=True),
                        warp(base, shift_y=8, shift_x=-6),
                        warp(base, degrees=45)):
        got, score = nearest_class(transformed)
        mark = " " if got == kind else "!"
        results.append("%-10s %.3f%s" % (SHAPE_NAMES[got], score, mark))
    print("%-11s %-22s %-22s %s" % (SHAPE_NAMES[kind], results[0], results[1], results[2]))
print()
print("знак ! означає, що мітка змінилася — такий приклад пішов би в навчання з брехнею")

Дзеркало й зсув не змінили жодної мітки. Поворот на 45° змінив чотири з шести — і
найяскравіше на квадраті: він **точно** перетворився на ромб.

Тепер знайдемо, з якого кута мітка ламається для кожного класу. Це число і є межею
безпечної сили повороту в цьому датасеті.

In [ ]:
print("%-11s %s" % ("клас", "найменший кут, на якому мітка вже не та"))
print("-" * 52)
break_angles = {}
for kind in range(6):
    base = shape_mask(kind, BASE_RADIUS)
    first_bad = None
    for degrees in range(0, 91):
        if nearest_class(warp(base, degrees=degrees))[0] != kind:
            first_bad = degrees
            break
    break_angles[kind] = first_bad
    print("%-11s %s" % (SHAPE_NAMES[kind], first_bad if first_bad is not None else "не ламається до 90°"))

# частка збережених міток, якщо кут беруть випадково з відрізка [-a, a]
keeps = np.zeros((6, 61), dtype=bool)
for kind in range(6):
    base = shape_mask(kind, BASE_RADIUS)
    for degrees in range(61):
        keeps[kind, degrees] = nearest_class(warp(base, degrees=degrees))[0] == kind

print()
print("максимальний кут → частка прикладів, де мітка вціліла")
for limit in (5, 8, 10, 15, 20, 30, 45, 60):
    print("   до %2d° : %.3f" % (limit, keeps[:, :limit + 1].mean()))

Ось і відповідь, ще до першого навчання: **поворот до 10° у цьому датасеті безпечний,
а поворот до 45° псує майже половину прикладів**. Далі перевіримо, чи погодиться з цим
мережа.

## 4 · Мережа й функція навчання

Мережа — та сама, що в темі 09: три блоки `Conv → ReLU → Pool` і голова з `Flatten`.
Нового тут лише одне: у навчальному циклі перед подачею в мережу батч проходить через
`transform`, якщо він заданий.

Важлива деталь про бюджет: усі досліди роблять **однакову кількість кроків** (250), а не
однакову кількість епох. Інакше мережа на 120 прикладах отримала б у десять разів менше
оновлень ваг, ніж на 1200, і ми порівнювали б не кількість даних, а кількість обчислень.

In [ ]:
STEPS = 250          # однаковий бюджет навчання для всіх дослідів
BATCH = 32
CHECK_EVERY = 25     # як часто міряти точність


def conv_block(in_channels, out_channels):
    """Один типовий блок із теми 09: Conv → ReLU → Pool."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class ShapeNet(nn.Module):
    """Три блоки з подвоєнням каналів і голова з Flatten. 24 774 параметри."""

    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(conv_block(1, 8), conv_block(8, 16), conv_block(16, 32))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 3 * 3, 64), nn.ReLU(), nn.Linear(64, 6))

    def forward(self, x):
        return self.head(self.body(x))


def accuracy(model, images, labels):
    """Частка правильних відповідей; градієнти тут не потрібні, тому no_grad."""
    model.eval()
    with torch.no_grad():
        parts = [model(images[s:s + 300]).argmax(dim=1) for s in range(0, len(images), 300)]
    model.train()
    return round((torch.cat(parts) == labels).float().mean().item(), 4)


def train_model(train_x, train_y, transform=None, steps=STEPS, seed=0):
    """Навчає мережу з нуля. Повертає модель, історію точності й витрачений час."""
    torch.manual_seed(seed)                       # без зерна числа не повторяться
    model = ShapeNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    loss_function = nn.CrossEntropyLoss()
    history = []
    started = time.time()

    order = torch.randperm(len(train_x))
    position = 0
    for step in range(1, steps + 1):
        if position + BATCH > len(train_x):       # вибірка скінчилась — перемішуємо знову
            order = torch.randperm(len(train_x))
            position = 0
        batch_index = order[position:position + BATCH]
        position += BATCH

        batch_images = train_x[batch_index]
        if transform is not None:
            # аугментація на льоту: щоразу нові випадкові параметри
            batch_images = transform(batch_images)

        optimizer.zero_grad()
        loss_function(model(batch_images), train_y[batch_index]).backward()
        optimizer.step()

        if step % CHECK_EVERY == 0:
            history.append((step,
                            accuracy(model, train_x, train_y),
                            accuracy(model, test_x, test_y)))
    return model, history, round(time.time() - started, 1)


print("параметрів у мережі:", sum(p.numel() for p in ShapeNet().parameters()))

Конвеєр аугментації описується одним `Compose`. Різні досліди відрізнятимуться **рівно
одним числом** — максимальним кутом повороту.

In [ ]:
def augmentation(max_degrees):
    """Дзеркало, поворот, зсув і масштаб. Міняється тільки кут."""
    return v2.Compose([
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomAffine(degrees=max_degrees, translate=(0.2, 0.2), scale=(0.85, 1.15)),
    ])


# перевіримо відтворюваність: із тим самим зерном перетворення дає той самий результат
torch.manual_seed(123)
first_run = augmentation(8)(pool_x[:8])
torch.manual_seed(123)
second_run = augmentation(8)(pool_x[:8])
assert torch.equal(first_run, second_run), "аугментація не відтворюється!"
print("✅ з тим самим torch.manual_seed аугментація дає той самий результат")

## 5 · Замір 1: виграш на малій вибірці

Беремо 120 навчальних зображень (по 20 на клас) і навчаємо дві мережі: без аугментації
та з аугментацією, у якій поворот обмежено 8° — тим самим кутом, який геометрія щойно
визнала безпечним. Усе решта однакове, включно з зерном.

**Це перші два навчання з восьми; разом вони займають десятки секунд.**

In [ ]:
results = {}          # сюди складаємо всі досліди зошита

model, history, seconds = train_model(pool_x[:120], pool_y[:120], None)
results["без аугментації, 120"] = history
print("без аугментації, 120: %5.1f с  на навчальній %.3f  на перевірочній %.3f"
      % (seconds, history[-1][1], history[-1][2]))

model, history, seconds = train_model(pool_x[:120], pool_y[:120], augmentation(8))
results["з аугментацією, 120"] = history
print("з аугментацією,  120: %5.1f с  на навчальній %.3f  на перевірочній %.3f"
      % (seconds, history[-1][1], history[-1][2]))

Дивимось не на одне число, а на **розрив** між навчальною й перевірочною точністю. Саме
він і є мірою запамʼятовування.

In [ ]:
without = results["без аугментації, 120"]
with_aug = results["з аугментацією, 120"]

print("%6s | %-24s | %s" % ("крок", "без аугментації", "з аугментацією"))
print("%6s | %7s %7s %7s | %7s %7s %7s"
      % ("", "навч.", "перев.", "розрив", "навч.", "перев.", "розрив"))
print("-" * 62)
for (step, a_train, a_test), (_, b_train, b_test) in zip(without, with_aug):
    print("%6d | %7.3f %7.3f %7.3f | %7.3f %7.3f %7.3f"
          % (step, a_train, a_test, a_train - a_test, b_train, b_test, b_train - b_test))

print()
print("наприкінці навчання розрив без аугментації %.3f, з нею %.3f"
      % (without[-1][1] - without[-1][2], with_aug[-1][1] - with_aug[-1][2]))
print("виграш на перевірочній вибірці: %+.3f" % (with_aug[-1][2] - without[-1][2]))

## 6 · Замір 3: коли аугментація шкодить

Тепер шкідливий випадок. Беремо той самий конвеєр і той самий набір із 120 зображень,
міняємо **одне число** — максимальний кут повороту: 8° → 20° → 45°. Геометрія вже
сказала, чого чекати; подивимось, чи мережа з нею погодиться.

In [ ]:
for max_degrees in (20, 45):
    model, history, seconds = train_model(pool_x[:120], pool_y[:120], augmentation(max_degrees))
    results["поворот до %d°, 120" % max_degrees] = history
    print("поворот до %2d°: %5.1f с  на навчальній %.3f  на перевірочній %.3f"
          % (max_degrees, seconds, history[-1][1], history[-1][2]))

In [ ]:
print("%-18s %10s %11s %11s %13s"
      % ("аугментація", "мітка ціла", "навчальна", "перевірочна", "проти базової"))
print("-" * 68)
base_accuracy = results["без аугментації, 120"][-1][2]
rows = [("немає", 0, results["без аугментації, 120"][-1]),
        ("поворот до 8°", 8, results["з аугментацією, 120"][-1]),
        ("поворот до 20°", 20, results["поворот до 20°, 120"][-1]),
        ("поворот до 45°", 45, results["поворот до 45°, 120"][-1])]
for name, limit, last in rows:
    survived = keeps[:, :limit + 1].mean()
    print("%-18s %9.0f%% %11.3f %11.3f %+13.3f"
          % (name, survived * 100, last[1], last[2], last[2] - base_accuracy))

Числа виявились не такими простими, як обіцяла геометрія, і це найцінніше місце теми.
Поворот до 8°, де мітку зберігають **усі** приклади, допоміг. Поворот до 20°, де мітку
зберігають лише 84 %, допоміг **ще більше**. І тільки поворот до 45°, де брехнею стає майже
половина прикладів, виявився гіршим за повну відсутність аугментації.

Висновок: геометричний рахунок — це **попередження, а не заборона**. Він каже, скільки
прикладів стануть брехливими, але не каже, чи буде від цього шкода: трохи зіпсованих міток
мережа витримує, якщо взамін бачить більше різних картинок. Силу добирають замірами, а
геометрію використовують, щоб знати, де взагалі є сенс шукати.

## 7 · Замір 2: чи лишається виграш, коли даних більше

Повторюємо перший дослід на 360 і на 1200 прикладах. Бюджет навчання той самий — 250
кроків, — тож єдине, що змінюється, це кількість **різних** зображень, які бачить мережа.

**Це чотири навчання — приблизно половина всього часу зошита.**

In [ ]:
models = {}           # знадобляться в замірі 4

for count in (360, 1200):
    model, history, seconds = train_model(pool_x[:count], pool_y[:count], None)
    results["без аугментації, %d" % count] = history
    models["без аугментації, %d" % count] = model
    print("без аугментації, %4d: %5.1f с  на навчальній %.3f  на перевірочній %.3f"
          % (count, seconds, history[-1][1], history[-1][2]))

    model, history, seconds = train_model(pool_x[:count], pool_y[:count], augmentation(8))
    results["з аугментацією, %d" % count] = history
    models["з аугментацією, %d" % count] = model
    print("з аугментацією,  %4d: %5.1f с  на навчальній %.3f  на перевірочній %.3f"
          % (count, seconds, history[-1][1], history[-1][2]))

In [ ]:
print("%10s %14s %14s %10s" % ("навчальних", "без аугментації", "з аугментацією", "виграш"))
print("-" * 52)
for count in (120, 360, 1200):
    without_value = results["без аугментації, %d" % count][-1][2]
    with_value = results["з аугментацією, %d" % count][-1][2]
    print("%10d %14.3f %14.3f %+10.3f" % (count, without_value, with_value, with_value - without_value))

Виграш падає зі зростанням вибірки. Причину видно в колонці «на навчальній» попередніх
дослідів: з аугментацією мережа за ті самі 250 кроків встигає вивчити менше, бо задача
складніша. Коли даних мало, ця ціна окупається з надлишком; коли даних досить — окупати
вже нема чого.

Осторога до цього заміру: бюджет навчання тут фіксований. Якби ми вчили довше, картина на
найбільшій вибірці могла б змінитись. Висновок точний саме в постановці «однакова кількість
обчислень» — тій, у якій опиняється більшість практичних задач.

## 8 · Замір 4: що буде, якщо аугментувати перевірочну вибірку

Візьмемо дві моделі, які на чистій перевірочній вибірці показують майже однакову якість:
одна навчена на 1200 прикладах **без** аугментації, друга — на 360 **з** аугментацією.
На чистій вибірці кожна дає одне число, і друге вимірювання дасть те саме.

Тепер поміряємо їх вісім разів на **аугментованій** перевірочній вибірці, міняючи лише
зерно. Дивимось на дві речі: наскільки стрибає число в кожної моделі й чи міняється
відповідь на питання «яка модель краща».

In [ ]:
model_a = models["без аугментації, 1200"]     # модель А: багато даних, без аугментації
model_b = models["з аугментацією, 360"]       # модель Б: утричі менше даних, але з нею

clean_a = accuracy(model_a, test_x, test_y)
clean_b = accuracy(model_b, test_x, test_y)
print("на чистій перевірочній: А %.3f, Б %.3f" % (clean_a, clean_b))
print("різниця між моделями: %.3f — і вона повторювана" % abs(clean_a - clean_b))
print()

test_transform = augmentation(8)
runs_a, runs_b = [], []
for seed in range(8):
    torch.manual_seed(1000 + seed)
    # аугментуємо перевірочну вибірку так само, як навчальну — саме цього робити не можна
    noisy_test = torch.cat([test_transform(test_x[s:s + 64]) for s in range(0, len(test_x), 64)])
    runs_a.append(accuracy(model_a, noisy_test, test_y))
    runs_b.append(accuracy(model_b, noisy_test, test_y))
    print("зерно %4d: А → %.3f   Б → %.3f   краща: %s"
          % (1000 + seed, runs_a[-1], runs_b[-1],
             "А" if runs_a[-1] > runs_b[-1] else ("Б" if runs_b[-1] > runs_a[-1] else "нічия")))

In [ ]:
spread_a = max(runs_a) - min(runs_a)
spread_b = max(runs_b) - min(runs_b)
wins_a = sum(1 for a, b in zip(runs_a, runs_b) if a > b)
wins_b = sum(1 for a, b in zip(runs_a, runs_b) if b > a)

print("розкид від зерна: у моделі А — %.3f, у моделі Б — %.3f" % (spread_a, spread_b))
print("різниця між моделями на чистій вибірці: %.3f" % abs(clean_a - clean_b))
print("на аугментованій «кращою» була А в %d прогонах із 8, Б — у %d" % (wins_a, wins_b))
print()
print("середнє на аугментованій: А %.3f, Б %.3f"
      % (sum(runs_a) / len(runs_a), sum(runs_b) / len(runs_b)))
print("тобто зміна способу перевірки посунула відповідь на %.3f — більше, ніж уся"
      % abs((sum(runs_a) / len(runs_a)) - (sum(runs_b) / len(runs_b))))
print("різниця між моделями на чистій вибірці (%.3f)" % abs(clean_a - clean_b))

Два висновки, і другий важливіший за перший.

**Перший:** число на аугментованій перевірці плаває. Модель не змінилась, вибірка не
змінилась — змінилось лише зерно, а точність інша. Публікувати таке число як «точність
моделі» не можна.

**Другий:** аугментована перевірка не нейтральна. Вона міняє відповідь на питання «яка
модель краща» — а моделі при цьому ті самі, що й були. Тобто ти більше не порівнюєш моделі,
ти порівнюєш те, наскільки кожна з них дружить із твоїм конвеєром перетворень.

## 9 · TTA: аугментація на прогнозі — це інше

Помилка з розділу 8 полягала в тому, що випадковий варіант **заміняв** оригінал. TTA
робить протилежне: проганяє кілька варіантів і **усереднює** прогнози. Результат
детермінований, а ціна пряма — стільки прогонів мережі, скільки варіантів.

Але є умова, про яку легко забути: TTA допомагає лише тоді, коли модель **терпить** ці
перетворення. Перевіримо на обох моделях із попереднього розділу — на тій, що навчалась
без аугментації, і на тій, що з нею.

In [ ]:
def predict_with_tta(model, variants=4):
    """Усереднює ймовірності по оригіналу й кількох аугментованих варіантах."""
    torch.manual_seed(7)                      # набір варіантів має бути відтворюваним
    with torch.no_grad():
        # перший доданок — сам оригінал, далі аугментовані варіанти
        total_probabilities = torch.softmax(model(test_x), dim=1)
        for variant in range(variants):
            noisy_test = torch.cat([test_transform(test_x[s:s + 64])
                                    for s in range(0, len(test_x), 64)])
            total_probabilities = total_probabilities + torch.softmax(model(noisy_test), dim=1)
    return round((total_probabilities.argmax(dim=1) == test_y).float().mean().item(), 4)


started = time.time()
print("%-34s %10s %10s %8s" % ("модель", "звичайно", "з TTA", "різниця"))
print("-" * 66)
for name, model, clean_value in (("А · 1200 без аугментації", model_a, clean_a),
                                 ("Б · 360 з аугментацією", model_b, clean_b)):
    tta_value = predict_with_tta(model)
    print("%-34s %10.3f %10.3f %+8.3f" % (name, clean_value, tta_value, tta_value - clean_value))
print()
print("ціна в обох випадках однакова: пʼять прогонів мережі замість одного")
print("час на обидва TTA: %.1f с" % (time.time() - started))

## 10 · Завдання

### 🟢 Рівень 1

Повтори замір 1, замінивши афінне перетворення на
`v2.RandomResizedCrop(28, scale=(0.6, 1.0), antialias=True)`. Спершу поміряй геометрією, з
якого `scale` мітки починають ламатись, і лише потім навчай.

### 🟡 Рівень 2

Перебором по частці площі знайди, з якої частки `RandomErasing` ламає **кільце** — і
перевір це навчанням: дві мережі, з безпечним стиранням і з руйнівним.

### 🔴 Рівень 3

Зроби власний датасет зі стрілками (вгору, вниз, ліворуч, праворуч і дві подвійні), підбери
під нього набір перетворень і **доведи числом**, що кожне з них зберігає мітку.

Повний текст завдань із критеріями «зроблено» — у [homework.html](homework.html).